# Project 1: Differentiated Demand for Cars

## Theoretical Question 3 — Testing the Perloff-Salop Model Empirically

Perloff-Salop (1985) predicts a symmetric equilibrium price:

$$
p^* = c + \frac{1}{n(n-1)\int F(\varepsilon)^{n-2} f(\varepsilon)^2\,d\varepsilon}
$$

Price therefore depends only on marginal cost $c$ and the intensity of competition $n$, together with the shape of the match-value distribution $F$, which is treated as fixed/structural.

Crucially, price should **not** depend on a product's own observed characteristics such as weight or fuel efficiency. In the baseline Perloff-Salop model, firms are otherwise symmetric and differentiation works through the i.i.d. match value rather than through systematic vertical quality that would affect equilibrium pricing.

### Empirical test

We regress price on:

- a proxy for marginal cost,
- the number of competing firms $n$,
- taxes,
- weight (`we`), and
- fuel efficiency (`li`).

We then conduct a joint F-test of:

$$
H_0: \beta_{we}=0 \quad \text{and} \quad \beta_{li}=0.
$$

Rejecting $H_0$ provides evidence against the baseline Perloff-Salop pricing restriction.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm


In [2]:
# Load data
df = pd.read_csv("cars.csv")

# Define a market as a year-country combination
df["market"] = df["ye"].astype(str) + "_" + df["ma"].astype(str)

# Number of distinct competing firms in each market
# This is used as a proxy for n in the Perloff-Salop model.
df["n_firms"] = df.groupby("market")["frm"].transform("nunique")

df.head()


,ye,ma,co,zcode,brd,type,brand,model,org,loc,...,pop,ngdp,rgdp,engdp,ergdp,engdpc,ergdpc,inc,market,n_firms
0,70,1,15,14,2,audi 100/200,audi,100/200,2,4,...,9660000,1280999948288,3.940725e+12,25795239936,7.935358e+10,2670.314697,8214.65625,20363.7252,70_1,14
1,70,1,26,35,4,citroen 2 CV 6 - 2 CV 4,citroen,2CV6,1,3,...,9660000,1280999948288,3.940725e+12,25795239936,7.935358e+10,2670.314697,8214.65625,20363.7252,70_1,14
2,70,1,36,36,4,citroen dyane,citroen,dyane,1,3,...,9660000,1280999948288,3.940725e+12,25795239936,7.935358e+10,2670.314697,8214.65625,20363.7252,70_1,14
3,70,1,64,67,7,fiat 128,fiat,128,3,5,...,9660000,1280999948288,3.940725e+12,25795239936,7.935358e+10,2670.314697,8214.65625,20363.7252,70_1,14
4,70,1,71,80,8,ford escort,ford,escort,2,4,...,9660000,1280999948288,3.940725e+12,25795239936,7.935358e+10,2670.314697,8214.65625,20363.7252,70_1,14


In [3]:
# Cost proxy: producer price index of the exporting country (avppr)
# Remove observations for which logs cannot be constructed.
df = df[df["avppr"] > 0].dropna(
    subset=["avppr", "eurpr", "tax", "n_firms", "we", "li"]
).copy()

df = df[df["eurpr"] > 0].copy()

# Log transformations
df["log_avppr"] = np.log(df["avppr"])
df["log_eurpr"] = np.log(df["eurpr"])

df[["log_eurpr", "log_avppr", "tax", "n_firms", "we", "li"]].describe()


,log_eurpr,log_avppr,tax,n_firms,we,li
count,4867.000000,4867.000000,4867.000000,4867.000000,4867.000000,4867.000000
mean,8.849801,4.453633,0.210090,12.588042,946.421409,7.761427
std,0.579223,0.391688,0.054043,1.440063,199.329123,1.438906
min,7.116089,2.760704,0.130000,8.000000,555.000000,4.500000
25%,8.489058,4.372268,0.175000,12.000000,790.000000,6.733333
50%,8.868085,4.550809,0.206000,13.000000,920.000000,7.500000
75%,9.236532,4.620448,0.250000,14.000000,1070.000000,8.533334
max,10.355472,14.855201,0.330000,16.000000,1825.000000,15.500000


In [4]:
# Estimate the pricing regression
X = sm.add_constant(df[["log_avppr", "tax", "n_firms", "we", "li"]])
y = df["log_eurpr"]

# Standard errors are clustered at the market level.
model = sm.OLS(y, X).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["market"]}
)

print(model.summary())


                            OLS Regression Results                            
Dep. Variable:              log_eurpr   R-squared:                       0.763
Model:                            OLS   Adj. R-squared:                  0.763
Method:                 Least Squares   F-statistic:                     681.1
Date:                Sun, 27 Sep 2026   Prob (F-statistic):          1.09e-100
Time:                        12:08:25   Log-Likelihood:                -745.78
No. Observations:                4867   AIC:                             1504.
Df Residuals:                    4861   BIC:                             1542.
Df Model:                           5                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.4729      0.564      9.702      0.0

In [5]:
# Joint test of the Perloff-Salop restriction on observed characteristics
# H0: coefficient on weight = 0 AND coefficient on fuel efficiency = 0
ftest = model.f_test("we = 0, li = 0")

print("Joint test of H0: coef(we) = 0 and coef(li) = 0")
print(ftest)


Joint test of H0: coef(we) = 0 and coef(li) = 0
<F test: F=1234.4913892829231, p=1.8359104170303777e-93, df_denom=149, df_num=2>


### Interpretation

- If the joint test **fails to reject** $H_0$, the data do not provide evidence that weight and fuel efficiency systematically affect prices after controlling for the cost proxy, taxes, and the number of firms. This is consistent with the tested pricing restriction of the baseline Perloff-Salop model.
- If the joint test **rejects** $H_0$, observed product characteristics have explanatory power for prices. This is evidence against the baseline Perloff-Salop pricing restriction.

The test should be interpreted as a test of this particular empirical implication of the model rather than as a test of every feature of the Perloff-Salop framework.